In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)



In [ ]:
health = pd.read_csv("geocode_health_centre.csv")

nfhs = pd.read_excel("NFHS_5_India_Districts_Factsheet_Data.xls")

census = pd.read_csv("census2011.csv")

In [ ]:
print("Health :", health.shape)
print("NFHS :", nfhs.shape)
print("Census :", census.shape)

Health : (200429, 13)
NFHS : (707, 109)
Census : (610, 7)


In [ ]:
print("Health Missing Values")
print(health.isnull().sum())

print("\nNFHS Missing Values")
print(nfhs.isnull().sum())

print("\nCensus Missing Values")
print(census.isnull().sum())

Health Missing Values
state_name                0
district_name             0
subdistrict_name          0
facility_type             0
facility_name             0
facility_address     200411
latitude                  0
longitude                15
activeflag_c              0
notional_physical         0
location_type             0
type_of_facility          1
nin_n                     0
dtype: int64

NFHS Missing Values
district_names                                                                           0
state_ut                                                                                 0
number_of_households_surveyed                                                            0
number_of_women_age_15_49_years_interviewed                                              0
number_of_men_age_15_54_years_interviewed                                                0
                                                                                        ..
women_(age_30_49_years)_ever_under

In [ ]:
health = health.drop_duplicates()

nfhs = nfhs.drop_duplicates()

census = census.drop_duplicates()

print("Duplicates Removed")

Duplicates Removed


In [ ]:
def clean_columns(df):

    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")
        .str.replace("-", "_")
        .str.replace("/", "_")
    )

    return df

health = clean_columns(health)

nfhs = clean_columns(nfhs)

census = clean_columns(census)

In [ ]:
def clean_text(series):

    return (
        series.astype(str)
        .str.upper()
        .str.strip()
    )

health["state_name"] = clean_text(health["state_name"])
health["district_name"] = clean_text(health["district_name"])

nfhs["state_ut"] = clean_text(nfhs["state_ut"])
nfhs["district_names"] = clean_text(nfhs["district_names"])

census["state"] = clean_text(census["state"])
census["district"] = clean_text(census["district"])

In [ ]:
health["state_name"] = health["state_name"].replace({

    "A & N ISLANDS":"ANDAMAN AND NICOBAR ISLANDS",

    "ODISHA":"ORISSA",

    "DADRA & NAGAR HAVELI":"DADRA AND NAGAR HAVELI",

    "DAMAN & DIU":"DAMAN AND DIU",

    "JAMMU & KASHMIR":"JAMMU AND KASHMIR"

})

nfhs["state_ut"] = nfhs["state_ut"].replace({

    "ANDAMAN & NICOBAR ISLANDS":"ANDAMAN AND NICOBAR ISLANDS",

    "NCT OF DELHI":"DELHI",

    "MAHARASTRA":"MAHARASHTRA",

    "ODISHA":"ORISSA",

    "JAMMU & KASHMIR":"JAMMU AND KASHMIR"

})

In [ ]:
for df in [health, nfhs, census]:

    numeric = df.select_dtypes(include=np.number).columns

    df[numeric] = df[numeric].fillna(df[numeric].median())

    categorical = df.select_dtypes(include="object").columns

    df[categorical] = df[categorical].fillna("UNKNOWN")

In [ ]:
health_summary = (

    health

    .groupby(["state_name","district_name"])

    .size()

    .reset_index(name="health_centres")

)

health_summary.head()

,state_name,district_name,health_centres
0,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,46
1,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,52
2,ANDAMAN AND NICOBAR ISLANDS,SOUTH ANDAMAN,46
3,ANDHRA PRADESH,ANANTAPUR,719
4,ANDHRA PRADESH,CHITTOOR,776


In [ ]:
master = census.merge(

    health_summary,

    left_on=["state","district"],

    right_on=["state_name","district_name"],

    how="left"

)

In [ ]:
master.drop(

    columns=["state_name","district_name"],

    inplace=True

)

In [ ]:
master["health_centres"] = master["health_centres"].fillna(0)

In [ ]:
master["population"] = master["population"].str.replace(",", "").astype(float)

master["population_per_health_centre"] = (
    master["population"]
    /
    master["health_centres"].replace(0,np.nan)
)

master["population_per_health_centre"] = (
    master["population_per_health_centre"]
    .fillna(master["population"])
)

In [ ]:
master["healthcare_density"] = pd.cut(

    master["population_per_health_centre"],

    bins=[0,5000,10000,20000,50000,10000000],

    labels=[

        "Excellent",

        "Good",

        "Average",

        "Poor",

        "Critical"

    ]

)

In [ ]:
master["healthcare_rank"] = (

    master["health_centres"]

    .rank(

        ascending=False,

        method="dense"

    )

)

In [ ]:
master.describe(include="all")

,ranking,district,state,population,growth,sex_ratio,literacy,health_centres,population_per_health_centre,healthcare_density,healthcare_rank
count,610.000000,610,610,6.100000e+02,610,610.000000,610.000000,610.000000,6.100000e+02,610,610.000000
unique,NaN,604,35,NaN,552,NaN,NaN,NaN,NaN,5,NaN
top,NaN,HAMIRPUR,UTTAR PRADESH,NaN,22.78 %,NaN,NaN,NaN,NaN,Good,NaN
freq,NaN,2,69,NaN,4,NaN,NaN,NaN,NaN,292,NaN
mean,329.598361,NaN,NaN,1.845019e+06,NaN,944.914754,72.344344,246.536066,2.761732e+05,NaN,213.416393
std,184.656903,NaN,NaN,1.566814e+06,NaN,61.247694,10.540881,204.643130,9.709090e+05,NaN,108.546693
min,1.000000,NaN,NaN,8.004000e+03,NaN,534.000000,36.100000,0.000000,1.020304e+03,NaN,1.000000
25%,183.250000,NaN,NaN,7.754178e+05,NaN,905.000000,65.255000,89.000000,4.420277e+03,NaN,123.250000
50%,335.500000,NaN,NaN,1.498161e+06,NaN,948.000000,72.150000,216.500000,6.264860e+03,NaN,222.500000
75%,487.750000,NaN,NaN,2.423572e+06,NaN,982.000000,79.987500,353.750000,9.064698e+03,NaN,309.000000


In [ ]:
master.to_csv(

    "master_healthcare_dataset.csv",

    index=False

)

print("Master Dataset Saved Successfully")

Master Dataset Saved Successfully


In [ ]:
print(master.shape)

master.head()

(610, 11)


,ranking,district,state,population,growth,sex_ratio,literacy,health_centres,population_per_health_centre,healthcare_density,healthcare_rank
0,1,THANE,MAHARASHTRA,11060148.0,36.01 %,886,84.53,336.0,3.291711e+04,Poor,134.0
1,2,NORTH TWENTY FOUR PARGANAS,WEST BENGAL,10009781.0,12.04 %,955,84.06,870.0,1.150550e+04,Average,7.0
2,3,BANGALORE,KARNATAKA,9621551.0,47.18 %,916,87.67,0.0,9.621551e+06,Critical,361.0
3,4,PUNE,MAHARASHTRA,9429408.0,30.37 %,915,86.15,773.0,1.219846e+04,Average,16.0
4,5,MUMBAI SUBURBAN,MAHARASHTRA,9356962.0,8.29 %,860,89.91,0.0,9.356962e+06,Critical,361.0
